# Day 10 — Notebook 01: Embeddings, Vector Databases & RAG
**LLM:** Qwen2.5-3B-Instruct (local)

This notebook builds the retrieval half of a grounded RAG system.

## Learning outcomes
- Explain embeddings and cosine similarity.
- Load and clean documents.
- Chunk documents with overlap.
- Store embeddings in ChromaDB.
- Retrieve top-k relevant chunks.
- Explain how chunk size affects retrieval quality.

In [ ]:
%pip install -q sentence-transformers chromadb pypdf transformers accelerate torch

In [ ]:
from pathlib import Path
from pypdf import PdfReader
import numpy as np

CORPUS = Path("../sample_corpus")
text_file = CORPUS / "ai_policy.txt"
text = text_file.read_text(encoding="utf-8")
print(text[:1500])

## 1. Chunking
We use a transparent word-based chunker. In production, compare fixed-size, recursive-character, sentence-based and semantic chunking.

In [ ]:
def chunk_text(text, chunk_size=120, overlap=25):
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap
    return chunks

chunks = chunk_text(text)
print("Chunks:", len(chunks))
for i, c in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i} ---\n{c[:500]}")

## 2. Embeddings and cosine similarity

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = embedding_model.encode(chunks, normalize_embeddings=True)

query = "What should an AI system do when it cannot answer from trusted information?"
q_vec = embedding_model.encode([query], normalize_embeddings=True)[0]
scores = embeddings @ q_vec
top_ids = np.argsort(scores)[::-1][:5]

for rank, idx in enumerate(top_ids, 1):
    print(f"Rank {rank} | score={scores[idx]:.3f} | chunk={idx}")
    print(chunks[idx][:400], "\n")

## 3. ChromaDB vector store

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection("day10_qwen_rag")

collection.upsert(
    ids=[f"chunk-{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=embeddings.tolist(),
    metadatas=[{"source":"ai_policy.txt","chunk":i} for i in range(len(chunks))]
)

result = collection.query(query_embeddings=[q_vec.tolist()], n_results=3)
for i, doc in enumerate(result["documents"][0], 1):
    print(f"\nRetrieved #{i}:\n{doc}")

## 4. Chunk-size experiment
Compare small, medium and large chunks. Inspect whether the answer context is complete or fragmented.

In [ ]:
for size, overlap in [(80,10), (120,25), (220,40)]:
    test_chunks = chunk_text(text, size, overlap)
    test_emb = embedding_model.encode(test_chunks, normalize_embeddings=True)
    test_scores = test_emb @ q_vec
    best = int(np.argmax(test_scores))
    print(f"chunk_size={size}, overlap={overlap}, chunks={len(test_chunks)}, best_score={test_scores[best]:.3f}")
    print(test_chunks[best][:250], "\n")

## 5. Qwen2.5 grounded generation
The first run downloads the local model.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype="auto", device_map="auto"
)

In [ ]:
def qwen_generate(question, context, max_new_tokens=300):
    messages = [
        {"role":"system","content":"Answer only from the supplied context. If the context is insufficient, say you do not have enough information."},
        {"role":"user","content":f"Context:\n{context}\n\nQuestion: {question}\n\nGive a grounded answer and mention the source chunks used."}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

retrieved = collection.query(query_embeddings=[q_vec.tolist()], n_results=3)
context = "\n\n".join(
    f"[Source chunk {i+1}] {d}" for i, d in enumerate(retrieved["documents"][0])
)
print(qwen_generate(query, context))

## Student challenge
1. Add a second document.
2. Add document/page/section metadata.
3. Test easy, ambiguous and out-of-scope queries.
4. Compare top-k = 2, 4 and 6.
5. Record one retrieval failure and propose a fix.